# Tutorial 03 — Engines: compile once, run many

`QarpEngine` is the only execution object you need. Its API splits work the
way an optimisation loop wants it split:

* **`engine.build(primitives)`** — flatten, transpile and compile every circuit
  once. Symbols stay symbolic.
* **`engine.run(symbol_map)`** — substitute values and simulate. Cheap; call it
  thousands of times.

This split is why variational loops are fast: the expensive compilation never
happens inside the loop.

In [ ]:
import numpy as np
from qarp.operators import QubitOperator
from qarp.blocks import HEABlock
from qarp.algorithms import Sampler, StateVector
from qarp.engines import QarpEngine

ansatz = HEABlock(4, 2, True, True, True, False).build()
H = QubitOperator("Z0 Z1") + QubitOperator("Z2 Z3") + QubitOperator("X0", 0.3)

expval = StateVector(bra=ansatz, operator=H, ket=ansatz)

engine = QarpEngine()
engine.build([expval])            # compile once...

for theta in (0.0, 0.3, 0.6):     # ...run many
    params = {s: theta for s in ansatz.symbols}
    print(f"theta={theta:.1f}  <H> = {engine.run(params)[0]:+.6f}")

## 1. Reproducibility

Shot noise comes from the engine's C++ RNG, not from numpy. Seed it at
construction — same seed, same shots:

In [ ]:
from qarp.blocks import SimpleBlock

bell = SimpleBlock(2)
bell.h(0)
bell.cx(0, 1)
bell.measure([(0, 0), (1, 1)])
bell.build()

for attempt in range(2):
    s = Sampler(ket=bell, n_shots=100)
    e = QarpEngine(seed=123)
    e.build([s])
    print(e.run()[0])

## 2. Parameter sweeps with `batch_run`

For many parameter sets over the same circuits, `batch_run` keeps the
simulation loop in C++:

In [ ]:
param_sets = [{s: t for s in ansatz.symbols} for t in np.linspace(0, 2 * np.pi, 9)]

engine = QarpEngine()
batch = engine.batch_run([expval], param_sets)
for ps, row in zip(param_sets, batch):
    print(f"theta={list(ps.values())[0]:+.3f}  <H> = {row[0]:+.6f}")

## 3. Analytic gradients

For `StateVector` expectation values the engine computes exact gradients by
adjoint backpropagation — cost ~2 simulations regardless of the number of
parameters (other primitives fall back to the parameter-shift rule):

In [ ]:
engine = QarpEngine()
engine.build([expval])

params = {s: 0.4 for s in ansatz.symbols}
grad = engine.run_gradient(params)[0]
print("dE/dtheta_k:", np.round(grad, 6))

## 4. Devices (a teaser)

Everything above ran on an ideal all-to-all simulator. The engine also accepts
a device description — qubit count, connectivity, gate set, noise model — and
then routes and rebases circuits during `build`, and simulates noise
trajectories during `run`:

```python
engine = QarpEngine(n_qubits=6, architecture=..., noise_model=..., gate_set=...)
```

That is its own topic — see `mwe_devices.ipynb` and `mwe_engines.ipynb`.

## Poke at it

* `expval.compiled_circuits` — what `build` produced
* `engine.run(...)` twice with the same seeded engine vs re-built engines
* time `engine.build` vs `engine.run` to feel the compile/run asymmetry

**Next:** tutorial_04_variational_loop — wire blocks + primitives + engine +
optimizer into your own VQE, then meet the built-in one.